In [1]:
# [1] 경로 설정 및 이전 세션 캐시 복원 — 반드시 첫 셀
import os, glob, shutil, subprocess, json, time, urllib.request
from pathlib import Path

WORK   = Path('/kaggle/working')
MODELS = WORK / 'ollama_models'
HFC    = WORK / 'hf_cache'
EXP    = WORK / 'exp'
for d in (MODELS, HFC, EXP): d.mkdir(parents=True, exist_ok=True)
os.environ['OLLAMA_MODELS'] = str(MODELS)
os.environ['HF_HOME']       = str(HFC)

def find_cached(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return Path(hits[0]) if hits else None

src = find_cached('ollama_models')
if src and not (MODELS / 'manifests').exists():
    if not (MODELS / 'blobs').exists():
        os.symlink(src / 'blobs', MODELS / 'blobs')      # 크고 불변 → 링크
    shutil.copytree(src / 'manifests', MODELS / 'manifests', dirs_exist_ok=True)
    print(f"모델 캐시 복원: {src}")
src = find_cached('hf_cache')
if src and not any(HFC.iterdir()):
    shutil.copytree(src, HFC, dirs_exist_ok=True)
    print(f"HF 캐시 복원: {src}")
print("OLLAMA_MODELS =", os.environ['OLLAMA_MODELS'])

OLLAMA_MODELS = /kaggle/working/ollama_models


In [2]:
# [2] Ollama, zstd, 파이썬 패키지 설치
subprocess.run('apt-get update -qq && apt-get install -y -qq zstd', shell=True)
if shutil.which('ollama') is None:
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
else:
    print("ollama 이미 설치됨")
subprocess.run('pip install -q rank-bm25 sentence-transformers', shell=True, check=True)
print("설치 완료")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


설치 완료


In [3]:
# [3] 데이터셋을 읽기전용 input 에서 쓰기가능한 작업폴더로 복사
cands = set()
for pat in ('corpus_final.jsonl', '25_rag_run.py'):
    for hit in glob.glob(f'/kaggle/input/**/{pat}', recursive=True):
        cands.add(Path(hit).parent)
assert cands, "데이터셋을 못 찾았습니다"
SRC = sorted(cands, key=lambda p: len(p.parts))[0]
print("원본:", SRC)

n = 0
for s in SRC.rglob('*'):
    if s.is_file():
        d = EXP / s.relative_to(SRC)
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(s, d); n += 1
os.chdir(EXP)
print(f"{n}개 파일 복사 → {EXP}")

원본: /kaggle/input/datasets/dkimrok/defense-rag-experiment
88개 파일 복사 → /kaggle/working/exp


In [4]:
# [4] 스크립트 최신본·필수 데이터 검증 — 여기서 막히면 실험 진행 금지
checks = {'25_rag_run.py':'set_ollama_host', '27_orchestrate.py':'ollama_url',
          '36_closedbook.py':'set_ollama_host', '37_snapshot.py':'start_watch',
          '22_coverage_engine.py':'mode volume', '24_build_index.py':'dedup_parts'}
bad = [f for f, key in checks.items()
       if not Path(f).exists() or key not in Path(f).read_text(encoding='utf-8')]
for f in checks:
    print(f"  {'구버전/없음' if f in bad else '최신본'}  {f}")

need = ['corpus_final.jsonl','question_final.jsonl','run_plan.json',
        'index/chunks.jsonl','index/embeddings.npy','index/bm25.pkl']
miss = [f for f in need if not Path(f).exists()]
print("\n변형 폴더:", sorted(d for d in os.listdir('.') if d.startswith('cov_')))
print("누락 파일:", miss or "없음")
assert not bad and not miss, "*** 구버전 또는 누락 — 데이터셋을 갱신하세요 ***"
print("\n검증 통과")

  최신본  25_rag_run.py
  최신본  27_orchestrate.py
  최신본  36_closedbook.py
  최신본  37_snapshot.py
  최신본  22_coverage_engine.py
  최신본  24_build_index.py

변형 폴더: ['cov_core', 'cov_periph', 'cov_random', 'cov_vol']
누락 파일: 없음

검증 통과


In [5]:
# [5] Ollama 서버 2개 기동 (GPU 0, GPU 1)
subprocess.run(['pkill', '-f', 'ollama serve'], check=False); time.sleep(3)

BASE = dict(OLLAMA_KEEP_ALIVE='-1', OLLAMA_MAX_LOADED_MODELS='1',
            OLLAMA_NUM_PARALLEL='1', OLLAMA_CONTEXT_LENGTH='16384',
            OLLAMA_FLASH_ATTENTION='1',
            OLLAMA_MODELS=str(MODELS), HF_HOME=str(HFC))
SERVERS = {0: '127.0.0.1:11434', 1: '127.0.0.1:11435'}

def genv(gpu):
    e = os.environ.copy(); e.update(BASE)
    e['CUDA_VISIBLE_DEVICES'] = str(gpu); e['OLLAMA_HOST'] = SERVERS[gpu]
    return e

for gpu in SERVERS:
    lg = f'/kaggle/working/ollama_gpu{gpu}.log'
    subprocess.Popen(['ollama','serve'], stdout=open(lg,'w'),
                     stderr=subprocess.STDOUT, env=genv(gpu))
for gpu, host in SERVERS.items():
    for _ in range(90):
        try:
            urllib.request.urlopen(f'http://{host}/api/tags', timeout=2)
            print(f"GPU{gpu} 서버 기동 ({host})"); break
        except Exception: time.sleep(1)
    else:
        raise RuntimeError(f"GPU{gpu} 서버 미기동 — ollama_gpu{gpu}.log 확인")

GPU0 서버 기동 (127.0.0.1:11434)
GPU1 서버 기동 (127.0.0.1:11435)


In [6]:
# [6] 모델 준비 — GPU0에 8b, GPU1에 4b-instruct-2507(비사고형)
MODEL_8B = 'qwen3:8b'
MODEL_4B = 'kamekichi128/qwen3-4b-instruct-2507:latest'
ASSIGN   = {0: MODEL_8B, 1: MODEL_4B}

for gpu, model in ASSIGN.items():
    e = genv(gpu)
    have = subprocess.run(['ollama','list'], capture_output=True, text=True, env=e).stdout
    if model.split(':')[0] in have:
        print(f"GPU{gpu} {model} 이미 있음")
    else:
        print(f"GPU{gpu} {model} 내려받는 중...")
        subprocess.run(['ollama','pull',model], check=True, env=e)
    assert model.split(':')[0] in subprocess.run(
        ['ollama','list'], capture_output=True, text=True, env=e).stdout, \
        f"{model} 등록 실패"
print("모델 준비 완료")

GPU0 qwen3:8b 내려받는 중...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling a3de86cd1c13:   0% ▕                  ▏ 720 KB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   1% ▕                  ▏  43 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   2% ▕                  ▏  83 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   3% ▕                  ▏ 158 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   5% ▕                  ▏ 240 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   5% ▕                  ▏ 281 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   7% ▕█                 ▏ 365 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   9% ▕█                 ▏ 452 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   9% ▕█                 ▏ 494 MB/5.2 GB            

GPU1 kamekichi128/qwen3-4b-instruct-2507:latest 내려받는 중...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 3605803b982c:   2% ▕                  ▏  43 MB/2.5 GB                  pulling manifest 
pulling 3605803b982c:   4% ▕                  ▏  87 MB/2.5 GB                  pulling manifest 
pulling 3605803b982c:   7% ▕█                 ▏ 164 MB/2.5 GB                  pulling manifest 
pulling 3605803b982c:  10% ▕█                 ▏ 245 MB/2.5 GB                  pulling manifest 
pulling 3605803b982c:  11% ▕██                ▏ 284 MB/2.5 GB                  pulling manifest 
pulling 3605803b982c:  15% ▕██                ▏ 364 MB/2.5 GB                  pulling manifest 
pulling 3605803b982c:  18% ▕███               ▏ 441 MB/2.5 GB                  pulling manifest 
pulling 3605803b982c:  20% ▕███               ▏ 487 MB/2.5 GB                  pulling manifest 
pulling 3605803b982c:  23% ▕████              ▏ 565 MB

모델 준비 완료


pulling manifest 
pulling 3605803b982c: 100% ▕█████████████████ ▏ 2.5 GB/2.5 GB  411 MB/s      0s
verifying sha256 digest 
writing manifest 
success 


In [7]:
# [7] GPU 적재 게이트 — CPU 폴백이면 여기서 중단
for gpu, model in ASSIGN.items():
    e = genv(gpu)
    subprocess.run(['ollama','run',model,'안녕'], env=e, capture_output=True)
    ps = json.loads(urllib.request.urlopen(
        f'http://{SERVERS[gpu]}/api/ps', timeout=10).read())
    assert ps.get('models'), f"GPU{gpu} 적재 실패"
    for m in ps['models']:
        pct = m['size_vram']/m['size']*100 if m['size'] else 0
        print(f"GPU{gpu} {m['name']}: VRAM {pct:.1f}%")
        assert pct >= 99, f"*** GPU{gpu} CPU 폴백 — 진행 금지 ***"
print("두 GPU 모두 정상")

GPU0 qwen3:8b: VRAM 100.0%
GPU1 kamekichi128/qwen3-4b-instruct-2507:latest: VRAM 100.0%
두 GPU 모두 정상


In [8]:
# [8] 결과 자동 스냅샷 시작 + 질의 캐시 사전 생성
import importlib.util
spec = importlib.util.spec_from_file_location('snap', '37_snapshot.py')
snap = importlib.util.module_from_spec(spec); spec.loader.exec_module(snap)
snap.start_watch(['runs', 'runs_closedbook'], every=600)

# 캐시를 먼저 만들어 두면 두 프로세스가 동시에 만들려 경쟁하지 않는다
subprocess.run(['python','25_rag_run.py','index','question_final.jsonl',
                '--variant','cov_core/corpus_cov100_core_doc.jsonl','--dry-run'],
               check=True)

스냅샷 감시 시작: runs, runs_closedbook → /kaggle/working (600초 간격)
주의: 세션 자체가 끝나면 이 스냅샷도 사라집니다. 반드시 커밋하거나 zip 을 내려받으십시오.


질의 캐시 생성 중... (97문항 x 7,481청크)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2207.50it/s, Materializing param=pooler.dense.weight]


질의 캐시 저장: query_cache.npz (4.3MB, 77초)
변형 corpus_cov100_core_doc.jsonl | cov 100% core/doc
살아있는 청크 7,481 / 전체 7,481
벡터검색 ON | BM25 ON | k=5 alpha=0.5
서버 http://localhost:11434
모델 qwen3:8b | reasoning OFF(/no_think) | num_ctx=16384 num_predict=256 (dry-run)


문항 97개 처리 | 응답 0 | 실패 0
recall@5 (any-gold): 92% (74/80)
-> responses_cov100_core_doc_qwen3-8b.jsonl

[예시 검색] DAPA-L1-001
  - 방산물자 감손율 산정지침 적용범위
  - 방산물자 감손율 산정지침 목적
  - 군수품조달관리규정 적용범위
  - 방산물자 감손율 산정지침 업무분장
  - (방위사업청) 표창규정 적용범위


CompletedProcess(args=['python', '25_rag_run.py', 'index', 'question_final.jsonl', '--variant', 'cov_core/corpus_cov100_core_doc.jsonl', '--dry-run'], returncode=0)

In [9]:
# [9] 본실험 — 두 GPU에서 8b와 4b를 동시에 (약 3시간)
CMD = ['python','27_orchestrate.py','index','question_final.jsonl',
       '--cov-dirs','cov_core,cov_periph,cov_random,cov_vol',
       '--k','5','--num-ctx','16384','--timeout','120',
       '--run-plan','run_plan.json','--out','runs']

jobs = {}
for gpu, model in ASSIGN.items():
    lg = open(f'/kaggle/working/run_gpu{gpu}.log','w')
    jobs[gpu] = (subprocess.Popen(
        CMD + ['--models', model, '--ollama-url', f'http://{SERVERS[gpu]}'],
        stdout=lg, stderr=subprocess.STDOUT), lg)
    print(f"GPU{gpu} 시작: {model}")

while any(p.poll() is None for p,_ in jobs.values()):
    time.sleep(120)
    line = []
    for gpu,(p,_) in jobs.items():
        c = subprocess.run(f"grep -c '완료 (' /kaggle/working/run_gpu{gpu}.log",
                           shell=True, capture_output=True, text=True).stdout.strip()
        line.append(f"GPU{gpu} {'진행' if p.poll() is None else '종료'} {c}/25")
    print(time.strftime('%H:%M:%S'), ' | '.join(line))

for gpu,(p,lg) in jobs.items():
    lg.close(); print(f"GPU{gpu} 종료코드 {p.returncode}")
    print(open(f'/kaggle/working/run_gpu{gpu}.log').read()[-400:])

GPU0 시작: qwen3:8b
GPU1 시작: kamekichi128/qwen3-4b-instruct-2507:latest
07:32:29 GPU0 진행 0/25 | GPU1 진행 0/25
07:34:29 GPU0 진행 0/25 | GPU1 진행 0/25
07:36:29 GPU0 진행 0/25 | GPU1 진행 0/25
07:38:29 GPU0 진행 0/25 | GPU1 진행 0/25
[07:39:05] 스냅샷
  [스냅샷] snapshot_runs.zip (0.2MB, 파일 5개)
07:40:29 GPU0 진행 0/25 | GPU1 진행 0/25
07:42:29 GPU0 진행 0/25 | GPU1 진행 0/25
07:44:29 GPU0 진행 0/25 | GPU1 진행 0/25
07:46:29 GPU0 진행 0/25 | GPU1 진행 0/25
07:48:29 GPU0 진행 0/25 | GPU1 진행 0/25
[07:49:05] 스냅샷
  [스냅샷] snapshot_runs.zip (0.2MB, 파일 9개)
07:50:29 GPU0 진행 0/25 | GPU1 진행 0/25
07:52:29 GPU0 진행 0/25 | GPU1 진행 0/25
07:54:29 GPU0 진행 0/25 | GPU1 진행 0/25
07:56:29 GPU0 진행 0/25 | GPU1 진행 0/25
07:58:29 GPU0 진행 0/25 | GPU1 진행 0/25
[07:59:05] 스냅샷
  [스냅샷] snapshot_runs.zip (0.3MB, 파일 15개)
08:00:29 GPU0 진행 0/25 | GPU1 진행 0/25
08:02:29 GPU0 진행 0/25 | GPU1 진행 0/25
08:04:29 GPU0 진행 0/25 | GPU1 진행 0/25
08:06:29 GPU0 진행 0/25 | GPU1 진행 0/25
08:08:29 GPU0 진행 0/25 | GPU1 진행 0/25
[08:09:05] 스냅샷
  [스냅샷] snapshot_runs.zip (0.3MB, 파일 19개)
0

In [10]:
# [10] 폐쇄북 통제 arm — 두 GPU 동시 (약 10분)
jobs = {}
for gpu, model in ASSIGN.items():
    lg = open(f'/kaggle/working/cb_gpu{gpu}.log','w')
    jobs[gpu] = (subprocess.Popen(
        ['python','36_closedbook.py','question_final.jsonl','--model',model,
         '--ollama-url', f'http://{SERVERS[gpu]}','--out','runs_closedbook'],
        stdout=lg, stderr=subprocess.STDOUT), lg)
for gpu,(p,lg) in jobs.items():
    p.wait(); lg.close()
    print(open(f'/kaggle/working/cb_gpu{gpu}.log').read()[-800:])

서버 http://127.0.0.1:11434
폐쇄북 arm | 문항 97개 | 모델 qwen3:8b | num_ctx=16384 reasoning OFF

  [1번 문항 2초] 전체 예상 ~4분
  [적재] qwen3:8b VRAM 100.0% 정상(GPU)

문항 97개 | 응답 97 | 실패 0
확신도 분포: {'중간': 51, '모름': 13, '높음': 2, '<높음|중간|낮음|모름>': 31}
  (본실험은 전 조건 95~100% '높음' 이었다. 여기서 분산이 나오면 척도는 살아있고 검색 컨텍스트가 과신을 유발한 것이다.)
-> responses_closedbook_qwen3-8b.jsonl

서버 http://127.0.0.1:11435
폐쇄북 arm | 문항 97개 | 모델 kamekichi128/qwen3-4b-instruct-2507:latest | num_ctx=16384 reasoning OFF

  [1번 문항 2초] 전체 예상 ~3분
  [적재] kamekichi128/qwen3-4b-instruct-2507:latest VRAM 100.0% 정상(GPU)

문항 97개 | 응답 97 | 실패 0
확신도 분포: {'모름': 86, '중간': 8, '높음': 2, '낮음': 1}
  (본실험은 전 조건 95~100% '높음' 이었다. 여기서 분산이 나오면 척도는 살아있고 검색 컨텍스트가 과신을 유발한 것이다.)
-> responses_closedbook_kamekichi128-qwen3-4b-instruct-2507-latest.jsonl



In [11]:
# [11] 무결성 검증 및 핵심 지표 요약
import re, collections
char = {}
for mf in glob.glob('cov_*/coverage_manifest.json'):
    for c in json.load(open(mf, encoding='utf-8'))['conditions']:
        char[c['file'].replace('corpus_','').replace('.jsonl','')] = c['char_ratio']

def parse(a):
    a = (a or '').split('</think>')[-1]
    d = {}
    for ln in a.splitlines():
        for k in ('답','근거','확신도'):
            if ln.startswith(k): d[k] = ln.split(':',1)[-1].strip()
    return d
def abstain(a):
    t = parse(a).get('답','')
    return ('근거 없음' in t) or (t.strip() in ('없음','모름'))

print(f"{'조건':22s} {'모델':14s} {'문자%':>6s} {'응답':>4s} {'실패':>4s} "
      f"{'잘림':>4s} {'recall':>7s} {'높음%':>6s} {'기권%':>6s} {'OOS기권':>7s}")
print('-'*105)
rows_all = []
for f in sorted(glob.glob('runs/responses_*.jsonl')):
    rows = [json.loads(l) for l in open(f, encoding='utf-8')]
    if not rows: continue
    cond  = rows[0].get('condition','?')
    model = rows[0].get('model','?')
    mf = Path('runs')/f"meta_{Path(f).name[len('responses_'):-len('.jsonl')]}.json"
    meta = json.load(open(mf, encoding='utf-8')) if mf.exists() else {}
    hi  = sum(1 for r in rows if parse(r['raw_answer']).get('확신도')=='높음')
    ab  = sum(1 for r in rows if abstain(r['raw_answer']))
    oos = [r for r in rows if r['q_status']=='oos']
    rec = [r['recall_at_k'] for r in rows if r['recall_at_k'] is not None]
    print(f"{cond:22s} {model[:14]:14s} {char.get(cond,0)*100:5.1f}% {len(rows):4d} "
          f"{meta.get('n_error','?'):>4} {meta.get('n_trunc_suspect','?'):>4} "
          f"{(sum(rec)/len(rec)*100 if rec else 0):6.1f}% "
          f"{hi/len(rows)*100:5.1f}% {ab/len(rows)*100:5.1f}% "
          f"{sum(1 for r in oos if abstain(r['raw_answer'])):3d}/{len(oos):<3d}")
    rows_all.append(cond)

print(f"\n조건 파일 {len(rows_all)}개 (기대 60)")
for f in sorted(glob.glob('runs_closedbook/responses_*.jsonl')):
    rows = [json.loads(l) for l in open(f, encoding='utf-8')]
    conf = collections.Counter(parse(r['raw_answer']).get('확신도','[없음]') for r in rows)
    unk  = sum(1 for r in rows if '모름' in (parse(r['raw_answer']).get('답') or ''))
    print(f"\n[폐쇄북] {rows[0]['model']} | {len(rows)}건 "
          f"실패 {sum(1 for r in rows if r['error'])}")
    print(f"  확신도 {dict(conf)}")
    print(f"  '모름' 답변 {unk}/{len(rows)} ({unk/len(rows)*100:.0f}%)")

조건                     모델                문자%   응답   실패   잘림  recall    높음%    기권%   OOS기권
---------------------------------------------------------------------------------------------------------
cov0_core_doc          kamekichi128/q   0.0%   97    0    0   75.0%  21.6%  74.2%   9/17 
cov0_core_doc          qwen3:8b         0.0%   97    0    0   75.0%  95.9%  33.0%   4/17 
cov0_periph_doc        kamekichi128/q   0.0%   97    0    0    0.0%  21.6%  77.3%   9/17 
cov0_periph_doc        qwen3:8b         0.0%   97    0    0    0.0%  96.9%  34.0%   4/17 
cov0_random_doc        kamekichi128/q   0.0%   97    ?    ?    0.0%  21.6%  77.3%   9/17 
cov0_random_doc        qwen3:8b         0.0%   97    ?    ?    0.0%  96.9%  34.0%   4/17 
cov10_core_doc         kamekichi128/q   0.0%   97    0    0   85.7%  32.0%  63.9%   9/17 
cov10_core_doc         qwen3:8b         0.0%   97    0    0   85.7%  94.8%  28.9%   4/17 
cov10_periph_doc       kamekichi128/q   0.0%   97    0    0   68.8%  28.9%  70.1%   

In [12]:
# [12] 결과 보존 — 반드시 실행하고 Output 패널에서 zip 을 내려받을 것
snap.stop_watch()
snap.snapshot_once(['runs','runs_closedbook'])

n = len(glob.glob('runs/responses_*.jsonl'))
assert n == 60, f"응답 파일 {n}개 — 셀 9를 다시 실행하면 미완 조건부터 재개됩니다"

shutil.make_archive('/kaggle/working/ALL_RESULTS','zip','.', 'runs')
shutil.make_archive('/kaggle/working/ALL_CLOSEDBOOK','zip','.', 'runs_closedbook')
print(f"응답 파일 {n}개 | 저장 완료")
print(sorted(x for x in os.listdir('/kaggle/working') if x.endswith('.zip')))

스냅샷 감시 중지
  [스냅샷] snapshot_runs.zip (1.1MB, 파일 111개)
  [스냅샷] snapshot_runs_closedbook.zip (0.0MB, 파일 4개)
응답 파일 60개 | 저장 완료
['ALL_CLOSEDBOOK.zip', 'ALL_RESULTS.zip', 'snapshot_runs.zip', 'snapshot_runs_closedbook.zip']
